<h3>작은 인덱싱 비교 실험</h3>
<p>청크 크기와 임베딩 모델을 한 쌍으로 정해 처음부터 끝까지 실행합니다. 그다음 청킹만 바꾸고, 마지막으로 임베딩 모델만 바꾸어 결과를 기록합니다.</p>

In [1]:
# 실험에 사용할 LangChain 구성 요소를 현재 커널에 설치합니다.
%pip install -q "python-dotenv==1.2.3" "langchain-core==1.6.2"     "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import find_dotenv, load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 현재 노트북 폴더부터 상위 폴더에서 키 파일을 찾습니다.
import os

env_file = find_dotenv(usecwd=True)
assert env_file, "실습 최상위 폴더에 .env를 준비하세요."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY를 확인하세요."

text = """배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.

환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.

회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다."""
question = "제주 배송은 얼마나 더 걸리나요?"

<h3>1단계: 첫 조합을 끝까지 실행하기</h3>
<p>45자 재귀 청킹과 <code>text-embedding-3-small</code>을 사용합니다. 청크를 확인한 뒤 벡터스토어에 저장하고 질문으로 검색합니다.</p>

In [3]:
# 기준 설정으로 청크를 만들고 각 청크의 ID와 본문을 확인합니다.
baseline_splitter = RecursiveCharacterTextSplitter(
    chunk_size=45,
    chunk_overlap=8,
)
baseline_chunks = baseline_splitter.split_text(text)

baseline_documents = []
for index, chunk in enumerate(baseline_chunks):
    document = Document(
        page_content=chunk,
        metadata={"chunk_id": f"baseline:{index:02d}"},
    )
    baseline_documents.append(document)

for document in baseline_documents:
    print(document.metadata["chunk_id"], repr(document.page_content))

baseline:00 '배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.'
baseline:01 '환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.'
baseline:02 '회원 등급은 매월 갱신합니다. 포인트는 주문 금액의 1퍼센트가 적립됩니다.'


In [4]:
# 문서를 벡터스토어에 저장한 뒤 질문으로 검색해 본문과 점수를 확인합니다.
small_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=256,
)
baseline_store = InMemoryVectorStore(embedding=small_embeddings)
baseline_store.add_documents(baseline_documents)
baseline_results = baseline_store.similarity_search_with_score(question, k=2)

for document, score in baseline_results:
    print(document.metadata["chunk_id"], round(score, 4))
    print(document.page_content)

baseline:00 0.6453
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.
baseline:01 0.2405
환불은 수령 후 7일 이내 신청합니다. 사용한 상품은 환불할 수 없습니다.


<h3>2단계: 청킹만 바꾸기</h3>
<p>임베딩 모델과 질문은 유지하고 <code>chunk_size</code>만 45에서 30으로 줄입니다.</p>

In [5]:
# 모델과 질문은 유지하고 청크 크기만 45자에서 30자로 바꿉니다.
smaller_splitter = RecursiveCharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=8,
)
smaller_chunks = smaller_splitter.split_text(text)

smaller_documents = []
for index, chunk in enumerate(smaller_chunks):
    document = Document(
        page_content=chunk,
        metadata={"chunk_id": f"smaller:{index:02d}"},
    )
    smaller_documents.append(document)

print("기준 청크 수:", len(baseline_documents))
print("변경 청크 수:", len(smaller_documents))

기준 청크 수: 3
변경 청크 수: 6


In [6]:
# 임베딩 모델은 앞에서 만든 small_embeddings를 그대로 사용합니다.
smaller_store = InMemoryVectorStore(embedding=small_embeddings)
smaller_store.add_documents(smaller_documents)
smaller_results = smaller_store.similarity_search_with_score(question, k=2)

for document, score in smaller_results:
    print(document.metadata["chunk_id"], round(score, 4))
    print(document.page_content)

smaller:01 0.5631
배송은 기본 배송보다 이틀 더 걸립니다.
smaller:00 0.5625
배송은 결제 다음 날 시작합니다. 제주 배송은 기본


<h3>3단계: 임베딩 모델만 바꾸기</h3>
<p>30자 청크와 질문은 유지하고 모델만 <code>text-embedding-3-large</code>로 바꿉니다.</p>

In [7]:
# 청크와 질문은 유지하고 임베딩 모델만 변경해 검색합니다.
large_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=256,
)
large_store = InMemoryVectorStore(embedding=large_embeddings)
large_store.add_documents(smaller_documents)
large_results = large_store.similarity_search_with_score(question, k=2)

for document, score in large_results:
    print(document.metadata["chunk_id"], round(score, 4))
    print(document.page_content)

smaller:00 0.6148
배송은 결제 다음 날 시작합니다. 제주 배송은 기본
smaller:01 0.5452
배송은 기본 배송보다 이틀 더 걸립니다.


<h3>4단계: 관찰 결과 기록하기</h3>
<p>각 실행에서 바꾼 값, 청크 수, 첫 번째로 검색된 청크의 ID와 본문을 같은 이름의 딕셔너리 항목으로 기록하고 차례대로 출력합니다. 이 작은 결과만으로 최종 모델을 선택하지 않습니다.</p>

In [8]:
# 각 실행의 설정과 첫 번째 청크 본문을 함께 기록합니다.
comparison = []
comparison.append({
    "run": "기준", "chunk_size": 45, "model": "text-embedding-3-small",
    "chunks": len(baseline_documents),
    "top_chunk_id": baseline_results[0][0].metadata["chunk_id"],
    "top_content": baseline_results[0][0].page_content,
})

In [9]:
# 크기만 바꾼 결과도 같은 필드로 기록합니다.
comparison.append({
    "run": "청킹 변경", "chunk_size": 30, "model": "text-embedding-3-small",
    "chunks": len(smaller_documents),
    "top_chunk_id": smaller_results[0][0].metadata["chunk_id"],
    "top_content": smaller_results[0][0].page_content,
})

In [10]:
# 같은 30자 청크에 모델만 바꾼 결과를 기록합니다.
comparison.append({
    "run": "모델 변경", "chunk_size": 30, "model": "text-embedding-3-large",
    "chunks": len(smaller_documents),
    "top_chunk_id": large_results[0][0].metadata["chunk_id"],
    "top_content": large_results[0][0].page_content,
})

In [11]:
# 세 실행의 설정과 첫 청크 본문을 같은 필드 순서로 관찰합니다.
for row in comparison:
    print(row)


{'run': '기준', 'chunk_size': 45, 'model': 'text-embedding-3-small', 'chunks': 3, 'top_chunk_id': 'baseline:00', 'top_content': '배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.'}
{'run': '청킹 변경', 'chunk_size': 30, 'model': 'text-embedding-3-small', 'chunks': 6, 'top_chunk_id': 'smaller:01', 'top_content': '배송은 기본 배송보다 이틀 더 걸립니다.'}
{'run': '모델 변경', 'chunk_size': 30, 'model': 'text-embedding-3-large', 'chunks': 6, 'top_chunk_id': 'smaller:00', 'top_content': '배송은 결제 다음 날 시작합니다. 제주 배송은 기본'}


<h3>일부러 실패해 보기</h3>
<p>검색 점수가 있어도 어떤 본문을 맞는 결과로 볼지 정하지 않으면 검색이 정확했는지 확인할 수 없습니다.</p>

In [12]:
# 검색 점수만 있고 정답을 판단할 기준이 없는 상태를 확인합니다.
try:
    expected_keywords = None
    assert expected_keywords is not None, "정답 기준이 없습니다"
except AssertionError as error:
    print(type(error).__name__, str(error))
    print("현재는 검색 점수만 관찰할 수 있습니다.")
else:
    raise AssertionError("정답 기준이 없는 상태를 발견해야 합니다.")

AssertionError 정답 기준이 없습니다
현재는 검색 점수만 관찰할 수 있습니다.


<p><strong>수정 방법</strong> 제주 배송과 추가 기간에 관한 단서를 정하고 세 실행을 같은 기준으로 검사합니다. False가 나오면 해당 첫 결과에 단서가 빠졌다는 뜻입니다. 기준을 추가한 것과 검색 결과가 모두 정답인 것은 구분합니다.</p>

In [13]:
# 두 단서의 포함 여부를 같은 기준으로 판정합니다. 정답률 측정은 아닙니다.
expected_keywords = ["제주", "이틀"]
assert expected_keywords, "판정할 단서를 먼저 준비해야 합니다."
top_documents = [
    baseline_results[0][0], smaller_results[0][0], large_results[0][0],
]
# zip은 같은 실행의 기록과 첫 번째 검색 문서를 한 쌍으로 연결합니다.
for row, document in zip(comparison, top_documents):
    contains_all = True
    for keyword in expected_keywords:
        if keyword not in document.page_content:
            contains_all = False
    print(row["run"], "필요 단서 포함:", contains_all)
    print(document.page_content)
print("세 실행에 같은 판정 기준을 적용했습니다.")
print("False인 결과는 단서가 빠진 본문입니다. 청크 경계와 검색 결과를 확인합니다.")


기준 필요 단서 포함: True
배송은 결제 다음 날 시작합니다. 제주 배송은 기본 배송보다 이틀 더 걸립니다.
청킹 변경 필요 단서 포함: False
배송은 기본 배송보다 이틀 더 걸립니다.
모델 변경 필요 단서 포함: False
배송은 결제 다음 날 시작합니다. 제주 배송은 기본
세 실행에 같은 판정 기준을 적용했습니다.
False인 결과는 단서가 빠진 본문입니다. 청크 경계와 검색 결과를 확인합니다.
